In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
DATA_FOLDER = Path("../data")
print(list(DATA_FOLDER.glob("*.csv")))

[WindowsPath('../data/ACCESSCORP.csv'), WindowsPath('../data/DANGCEM.csv'), WindowsPath('../data/DANGSUGAR.csv'), WindowsPath('../data/GTCO.csv'), WindowsPath('../data/MTNN.csv'), WindowsPath('../data/NESTLE.csv'), WindowsPath('../data/PRESCO.csv'), WindowsPath('../data/SEPLAT.csv'), WindowsPath('../data/UBA.csv'), WindowsPath('../data/ZENITH.csv')]


In [2]:
import os
import sys
from pathlib import Path
import pandas as pd

#Seting directories 
NOTEBOOK_DIR = Path.cwd()
ROOT_DIR = NOTEBOOK_DIR.parent
SRC_DIR = ROOT_DIR / "src"
DATA_FOLDER = ROOT_DIR / "data"

SRC_DIR.mkdir(parents=True, exist_ok=True)

# Creating the 'data_loader.py' file inside 'src'
DATA_LOADER_CODE = """import pandas as pd
from pathlib import Path

def load_stock(file_path):
    \"\"\"Loads a stock CSV file, converts the Date column, and sorts it.\"\"\"
    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    return df
"""

with open(SRC_DIR / "data_loader.py", "w") as f:
    f.write(DATA_LOADER_CODE)
    
(SRC_DIR / "__init__.py").touch()
print("✅ Successfully created 'src/' folder, '__init__.py', and 'data_loader.py'!")

#Injecting the root path so Python can find our new module
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# 5. NOW we can import it perfectly!
from src.data_loader import load_stock

# 6. Load the stocks
all_stocks = { file.stem: load_stock(file) for file in DATA_FOLDER.glob("*.csv") }
print(f"🎉 Success! Loaded {len(all_stocks)} stocks into all_stocks.")

✅ Successfully created 'src/' folder, '__init__.py', and 'data_loader.py'!
🎉 Success! Loaded 10 stocks into all_stocks.


In [3]:
for ticker, df in all_stocks.items():
    print(
        f"{ticker:12} "
        f"Start: {df['Date'].min().date()} | "
        f"End: {df['Date'].max().date()} | "
        f"Rows: {len(df)}"
    )

ACCESSCORP   Start: 2022-03-29 | End: 2026-06-01 | Rows: 1030
DANGCEM      Start: 2020-06-01 | End: 2026-06-01 | Rows: 1485
DANGSUGAR    Start: 2020-06-01 | End: 2026-06-01 | Rows: 1485
GTCO         Start: 2020-06-01 | End: 2026-06-01 | Rows: 1481
MTNN         Start: 2020-06-01 | End: 2026-06-01 | Rows: 1485
NESTLE       Start: 2020-06-01 | End: 2026-06-01 | Rows: 1484
PRESCO       Start: 2020-06-01 | End: 2026-06-01 | Rows: 1469
SEPLAT       Start: 2020-06-01 | End: 2026-06-01 | Rows: 1486
UBA          Start: 2020-06-01 | End: 2026-06-01 | Rows: 1485
ZENITH       Start: 2020-06-01 | End: 2026-06-01 | Rows: 1486


In [4]:
START_DATE = "2022-06-01"
for ticker, df in all_stocks.items():
    all_stocks[ticker] = (
        df[df["Date"] >=START_DATE]
        .copy()
        .reset_index(drop=True)
    )

In [5]:
for ticker, df in all_stocks.items():
    print(
        f"{ticker:12}"
        f"Start: {df['Date'].min().date()} |"
        f"Rows: {len(df)}"
    )

ACCESSCORP  Start: 2022-06-01 |Rows: 988
DANGCEM     Start: 2022-06-01 |Rows: 988
DANGSUGAR   Start: 2022-06-01 |Rows: 988
GTCO        Start: 2022-06-01 |Rows: 988
MTNN        Start: 2022-06-01 |Rows: 988
NESTLE      Start: 2022-06-01 |Rows: 987
PRESCO      Start: 2022-06-01 |Rows: 972
SEPLAT      Start: 2022-06-01 |Rows: 989
UBA         Start: 2022-06-01 |Rows: 988
ZENITH      Start: 2022-06-01 |Rows: 989


In [6]:
# Clean Price Data & Recalculate Daily Returns
# ==========================================
for ticker, df in all_stocks.items():
    # 1. i Clean the 'Price' column: convert to string, remove commas, convert to float
    df["Price"] = (
        df["Price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .astype(float)
    )
    
   
    df["Daily_Return"] = df["Price"].pct_change()

print("✅ All stock prices cleaned and daily returns calculated successfully!")

✅ All stock prices cleaned and daily returns calculated successfully!


In [7]:
returns_list = []

for ticker, df in all_stocks.items():
    # Grab just the Date and the clean Daily_Return
    temp = (
        df[["Date", "Daily_Return"]]
        .rename(columns={"Daily_Return": ticker})
        .set_index("Date")
    )
    returns_list.append(temp)

# Concatenate all stocks along the columns axis
returns_df = pd.concat(returns_list, axis=1)


print("--- Missing Values Per Stock (Before Drop) ---")
print(returns_df.isna().sum())
print("-" * 46)

returns_df = returns_df.dropna()

print(f"\n🎉 Success! Final returns_df matrix shape: {returns_df.shape}")

--- Missing Values Per Stock (Before Drop) ---
ACCESSCORP     3
DANGCEM        3
DANGSUGAR      3
GTCO           3
MTNN           3
NESTLE         4
PRESCO        19
SEPLAT         2
UBA            3
ZENITH         2
dtype: int64
----------------------------------------------

🎉 Success! Final returns_df matrix shape: (969, 10)


In [8]:
returns_df.isna().sum()

ACCESSCORP    0
DANGCEM       0
DANGSUGAR     0
GTCO          0
MTNN          0
NESTLE        0
PRESCO        0
SEPLAT        0
UBA           0
ZENITH        0
dtype: int64

In [9]:
returns_df = returns_df.dropna()
returns_df.shape
returns_df.head()

,ACCESSCORP,DANGCEM,DANGSUGAR,GTCO,MTNN,NESTLE,PRESCO,SEPLAT,UBA,ZENITH
Date,,,,,,,,,,
2022-06-02,0.000000,0.0,-0.057143,-0.002212,0.000000,0.0,0.0,0.0,-0.018987,-0.019190
2022-06-03,0.005051,0.0,-0.030303,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.019565
2022-06-06,0.000000,0.0,0.000000,-0.002217,0.019565,0.0,0.0,0.0,0.006452,-0.004264
2022-06-07,0.000000,0.0,0.009375,-0.004444,0.023454,0.0,0.0,0.0,0.006410,0.002141
2022-06-08,-0.015075,0.0,0.006192,0.000000,0.000000,0.0,0.0,0.0,-0.006369,0.000000


In [10]:
TRADING_DAYS= 252
annual_returns = returns_df.mean()* TRADING_DAYS
annual_returns
#convert the annual returns to percentage
annual_returns*100

ACCESSCORP    35.719421
DANGCEM       41.990632
DANGSUGAR     46.311860
GTCO          53.678660
MTNN          41.346535
NESTLE        25.634108
PRESCO        74.568027
SEPLAT        56.078544
UBA           56.904050
ZENITH        52.716390
dtype: float64

In [11]:
#calaculating annual volatality
annual_volatility = returns_df.std() * np.sqrt(TRADING_DAYS)
annual_volatility
#higher volatility means the stocks flunctuate more

ACCESSCORP    0.430848
DANGCEM       0.345540
DANGSUGAR     0.539706
GTCO          0.363926
MTNN          0.366404
NESTLE        0.291492
PRESCO        0.333928
SEPLAT        0.296180
UBA           0.442655
ZENITH        0.384442
dtype: float64

In [12]:
returns_list = []
for ticker, df in all_stocks.items():
    temp = (
        df[["Date", "Daily_Return"]]
        .rename(columns={"Daily_Return": ticker})
        .set_index("Date")
    )
    returns_list.append(temp)

returns_df = pd.concat(returns_list, axis=1)

# Drop the very first row (which is NaN for all stocks)
returns_df = returns_df.dropna()
TRADING_DAYS = 252

# Now that returns_df contains actual floats, these will work perfectly!
annual_return = returns_df.mean() * TRADING_DAYS
annual_volatility = returns_df.std() * np.sqrt(TRADING_DAYS)

metrics = pd.DataFrame({
    "Annual Return": annual_return,
    "Annual Volatility": annual_volatility
})

# Display the clean table
metrics

,Annual Return,Annual Volatility
ACCESSCORP,0.357194,0.430848
DANGCEM,0.419906,0.345540
DANGSUGAR,0.463119,0.539706
GTCO,0.536787,0.363926
MTNN,0.413465,0.366404
NESTLE,0.256341,0.291492
PRESCO,0.745680,0.333928
SEPLAT,0.560785,0.296180
UBA,0.569040,0.442655
ZENITH,0.527164,0.384442


In [13]:
#saved the metrics
metrics.to_csv("../notebook/metric_s1.csv")
returns_df.describe()

,ACCESSCORP,DANGCEM,DANGSUGAR,GTCO,MTNN,NESTLE,PRESCO,SEPLAT,UBA,ZENITH
count,969.000000,969.000000,969.000000,969.000000,969.000000,969.000000,969.000000,969.000000,969.000000,969.000000
mean,0.001417,0.001666,0.001838,0.002130,0.001641,0.001017,0.002959,0.002225,0.002258,0.002092
std,0.027141,0.021767,0.033998,0.022925,0.023081,0.018362,0.021036,0.018658,0.027885,0.024218
min,-0.142222,-0.100000,-0.100000,-0.119403,-0.100000,-0.100000,-0.099983,-0.100000,-0.113314,-0.120032
25%,-0.009009,0.000000,-0.005556,-0.005738,0.000000,0.000000,0.000000,0.000000,-0.008264,-0.006311
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.010638,0.000000,0.004808,0.009174,0.000000,0.000000,0.000000,0.000000,0.012474,0.010035
max,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100002,0.100000,0.100000


In [14]:
annual_return = (
    (1+ returns_df).prod()
)**(252 / len(returns_df))- 1
metrics["VaR (95%)"] = returns_df.quantile(0.05)

In [16]:
RISK_FREE_RATE = 0.10  # 10% annual risk-free rate

# We Calculate the Sharpe Ratio first so the column exists!

metrics["VaR (95%)"] = returns_df.quantile(0.05)

metrics["Return Rank"] = metrics["Annual Return"].rank(ascending=False)
metrics["Risk Rank"] = metrics["Annual Volatility"].rank(ascending=True)
metrics["Sharpe Rank"] = metrics["Sharpe Ratio"].rank(ascending=False)

metrics = metrics.sort_values(by="Sharpe Rank")
metrics

KeyError: 'Sharpe Ratio'

In [17]:
#calculating for maximum drawdown
# Maximum Drawdown is the largest percentage decline from a previous peak.
def max_drawdown(returns):
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    return drawdown.min()

metrics["Maximum Drawdown"] = returns_df.apply(max_drawdown)
metrics

,Annual Return,Annual Volatility,VaR (95%),Return Rank,Risk Rank,Maximum Drawdown
ACCESSCORP,0.357194,0.430848,-0.037315,9.0,8.0,-0.466667
DANGCEM,0.419906,0.345540,0.000000,7.0,4.0,-0.495413
DANGSUGAR,0.463119,0.539706,-0.052649,6.0,10.0,-0.654567
GTCO,0.536787,0.363926,-0.029860,4.0,5.0,-0.383600
MTNN,0.413465,0.366404,-0.031198,8.0,6.0,-0.427119
NESTLE,0.256341,0.291492,0.000000,10.0,1.0,-0.419842
PRESCO,0.745680,0.333928,0.000000,1.0,3.0,-0.398907
SEPLAT,0.560785,0.296180,0.000000,3.0,2.0,-0.265991
UBA,0.569040,0.442655,-0.039947,2.0,9.0,-0.434462
ZENITH,0.527164,0.384442,-0.031070,5.0,7.0,-0.366391


In [18]:
#calculating the correlation matrx
correlation_matrix = returns_df.corr()
correlation_matrix

,ACCESSCORP,DANGCEM,DANGSUGAR,GTCO,MTNN,NESTLE,PRESCO,SEPLAT,UBA,ZENITH
ACCESSCORP,1.000000,0.072701,0.240758,0.435660,0.187673,0.097543,0.058458,0.054721,0.565080,0.521188
DANGCEM,0.072701,1.000000,0.039234,0.046981,0.110888,0.014738,0.042499,0.022322,0.020932,0.083531
DANGSUGAR,0.240758,0.039234,1.000000,0.268197,0.114855,0.050850,0.084428,0.012889,0.216688,0.221336
GTCO,0.435660,0.046981,0.268197,1.000000,0.193143,0.077873,0.121512,0.046831,0.404036,0.581175
MTNN,0.187673,0.110888,0.114855,0.193143,1.000000,0.063759,0.037270,0.057855,0.164439,0.230354
NESTLE,0.097543,0.014738,0.050850,0.077873,0.063759,1.000000,0.046231,0.072320,0.095058,0.089813
PRESCO,0.058458,0.042499,0.084428,0.121512,0.037270,0.046231,1.000000,0.014667,0.027632,0.052895
SEPLAT,0.054721,0.022322,0.012889,0.046831,0.057855,0.072320,0.014667,1.000000,0.088106,0.056950
UBA,0.565080,0.020932,0.216688,0.404036,0.164439,0.095058,0.027632,0.088106,1.000000,0.542169
ZENITH,0.521188,0.083531,0.221336,0.581175,0.230354,0.089813,0.052895,0.056950,0.542169,1.000000


In [19]:
#save the corr matrix
correlation_matrix.to_csv("../notebook/crrelation_matrix.csv")

In [20]:
#created a sector map... this project requires a sector beta , so we can identify which stock belong to which sector
sector_map = {
    "ACCESSCORP": "Banking",
    "GTCO": "Banking",
    "ZENITH": "Banking",
    "DANGCEM": "Industrial",
    "DANGSUGAR": "Consumer Goods",
    "NESTLE": "Consumer Goods",
    "MTNN" : "Telecommunications",
    "SEPLAT": "Oil & Gas",
    "PRESCO": "Agriculture",
    "UBA": "Banking"
    
}
metrics["Sector"] = metrics.index.map(sector_map)
metrics.to_csv("../notebook/stocks_metrics.csv", index = True)
metrics

,Annual Return,Annual Volatility,VaR (95%),Return Rank,Risk Rank,Maximum Drawdown,Sector
ACCESSCORP,0.357194,0.430848,-0.037315,9.0,8.0,-0.466667,Banking
DANGCEM,0.419906,0.345540,0.000000,7.0,4.0,-0.495413,Industrial
DANGSUGAR,0.463119,0.539706,-0.052649,6.0,10.0,-0.654567,Consumer Goods
GTCO,0.536787,0.363926,-0.029860,4.0,5.0,-0.383600,Banking
MTNN,0.413465,0.366404,-0.031198,8.0,6.0,-0.427119,Telecommunications
NESTLE,0.256341,0.291492,0.000000,10.0,1.0,-0.419842,Consumer Goods
PRESCO,0.745680,0.333928,0.000000,1.0,3.0,-0.398907,Agriculture
SEPLAT,0.560785,0.296180,0.000000,3.0,2.0,-0.265991,Oil & Gas
UBA,0.569040,0.442655,-0.039947,2.0,9.0,-0.434462,Banking
ZENITH,0.527164,0.384442,-0.031070,5.0,7.0,-0.366391,Banking


In [24]:
# to create a sectr return... since i dont haev a seperate secto index,
#w'll create one by taking the average daily return of all stocks in each sector
sector_returns  = pd.DataFrame(index= returns_df.index)
for sector in set(sector_map.values()):
    stocks = [s for s,sec in sector_map.items() if sec == sector]
    sector_returns[sector] = returns_df[stocks].mean(axis = 1)

 #calcualting the beta
def calculate_beta(stock_returns, benchmark_returns):
    covariance = np.cov(stock_returns, benchmark_returns)[0,1]
    variance = np.var(benchmark_returns)
    return covariance / variance 

    
#bcompputing beta for each stock

betas = {}

for stock in returns_df.columns:
    sector = sector_map[stock]
    benchmark = sector_returns[sector]

    betas[stock] = calculate_beta(
        returns_df[stock],
        benchmark
    )
RISK_FREE_RATE = 0.10

metrics["Sharpe Ratio"] = (
    metrics["Annual Return"] - RISK_FREE_RATE
) / metrics["Annual Volatility"]

metrics["Sector Beta"] = pd.Series(betas)
metrics = metrics.reset_index()
metrics = metrics.rename(columns= {"index": "Stocks"})
metrics.to_csv("../notebook/final_stock_metrics.csv", index=False)
metrics

,Stocks,Stocks,Annual Return,Annual Volatility,VaR (95%),Return Rank,Risk Rank,Maximum Drawdown,Sector,Sector Beta,Sharpe Ratio
0,0,ACCESSCORP,0.357194,0.430848,-0.037315,9.0,8.0,-0.466667,Banking,NaN,0.596949
1,1,DANGCEM,0.419906,0.345540,0.000000,7.0,4.0,-0.495413,Industrial,NaN,0.925815
2,2,DANGSUGAR,0.463119,0.539706,-0.052649,6.0,10.0,-0.654567,Consumer Goods,NaN,0.672808
3,3,GTCO,0.536787,0.363926,-0.029860,4.0,5.0,-0.383600,Banking,NaN,1.200207
4,4,MTNN,0.413465,0.366404,-0.031198,8.0,6.0,-0.427119,Telecommunications,NaN,0.855518
5,5,NESTLE,0.256341,0.291492,0.000000,10.0,1.0,-0.419842,Consumer Goods,NaN,0.536347
6,6,PRESCO,0.745680,0.333928,0.000000,1.0,3.0,-0.398907,Agriculture,NaN,1.933589
7,7,SEPLAT,0.560785,0.296180,0.000000,3.0,2.0,-0.265991,Oil & Gas,NaN,1.555760
8,8,UBA,0.569040,0.442655,-0.039947,2.0,9.0,-0.434462,Banking,NaN,1.059607
9,9,ZENITH,0.527164,0.384442,-0.031070,5.0,7.0,-0.366391,Banking,NaN,1.111128


These mean: there is a 5% chance that the stock could lose atleast 2.99% (GTCO) or 3.99% (UBA) in a single trading day, based on historical data.
for the result with 0.0 VAR(e.g PRESCO) it likrly because because many daily returns are exactly zero (common for less frequently traded stocks on the NGX).

a beta close to 1 is expected because the stock is efectively being conppared to itsef.

PLEASE REFER TO MY README FOR MORE DETAIL...